# E02

In [ ]:

from __future__ import annotations

import csv
import json
import sys
from copy import deepcopy
from pathlib import Path


import shutil
import tempfile
import zipfile

SOURCE_ZIP_NAME = 'syniscopy_source.zip'
IN_COLAB = Path('/content').exists()
DRIVE_MYDRIVE = Path('/content/drive/MyDrive')
if IN_COLAB and not DRIVE_MYDRIVE.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive was not mounted automatically:', repr(exc))


def find_supplemental_root() -> Path:
    explicit = globals().get('SYNISCOPY_SUPPLEMENTAL_ROOT', None)
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])
    if DRIVE_MYDRIVE.exists():
        candidates.extend([
            DRIVE_MYDRIVE / 'supplemental',
            DRIVE_MYDRIVE / 'SyniscopySupplemental',
        ])
    candidates.extend([
        Path('/content/supplemental'),
        Path('/content/SyniscopySupplemental'),
    ])
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if (resolved / 'supplemental' / 'E01.ipynb').exists():
            resolved = resolved / 'supplemental'
        if (resolved / 'E01.ipynb').exists() and (
            (resolved / SOURCE_ZIP_NAME).exists() or (resolved.parent / 'codebase').is_dir()
        ):
            return resolved
    raise RuntimeError(
        'Syniscopy supplemental folder not found. Upload the supplemental folder to Drive as MyDrive/supplemental, '
        'including syniscopy_source.zip.'
    )


def prepare_syniscopy_source(supplemental_root: Path) -> Path:
    source_root = supplemental_root.parent
    if (source_root / 'codebase').is_dir():
        return source_root
    zip_path = supplemental_root / SOURCE_ZIP_NAME
    if not zip_path.exists():
        raise FileNotFoundError(
            f'Missing {zip_path}. Run python supplemental/package_experiments_for_colab.py before uploading supplemental/.'
        )
    extract_dir = Path('/content/syniscopy_source') if IN_COLAB else Path(tempfile.gettempdir()) / 'syniscopy_source'
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    if (extract_dir / 'codebase').is_dir():
        return extract_dir
    candidates = [p for p in extract_dir.iterdir() if p.is_dir() and (p / 'codebase').is_dir()]
    if len(candidates) != 1:
        raise RuntimeError(f'Could not find a single codebase/ root after extracting {zip_path}.')
    return candidates[0]


SUPPLEMENTAL_ROOT = find_supplemental_root()
ROOT = prepare_syniscopy_source(SUPPLEMENTAL_ROOT)
CODEBASE = ROOT / 'codebase'
OUTPUT_DIR = SUPPLEMENTAL_ROOT / 'outputs' / 'E02'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(CODEBASE) not in sys.path:
    sys.path.insert(0, str(CODEBASE))

import numpy as np

from composite_shapes import bent_trimer, component, dimer, linear_trimer, particle as composite_particle, rod_stack
from config import PARAMS
from fisher_diagnostic import compute_localization_orientation_crlb, predict_se3_rank_from_symmetry
from main import generate_single_frame_views

IMAGE_SIZE = 160
PUPIL_SAMPLES = 160
PIXEL_SIZE_NM = 65.0
WAVELENGTH_NM = 532.0
DIAMETER_NM = 100.0
Z_NM = 0.0
NOISE_VARIANCE = 1.0




def finite_or_inf(value: float, digits: int = 3) -> str:
    value = float(value)
    if not np.isfinite(value):
        return '$\\infty$'
    return f'{value:.{digits}g}'


def stable_seed(label: str) -> int:
    seed = 0
    for index, char in enumerate(str(label)):
        seed = (seed + (index + 1) * ord(char)) % (2**32)
    return seed


def display_from_contrast(contrast: np.ndarray, label: str) -> np.ndarray:
    image = np.asarray(contrast, dtype=float)
    finite = image[np.isfinite(image)]
    if finite.size == 0:
        return np.zeros(image.shape, dtype=np.uint8)
    center = float(np.median(finite))
    deviation = np.where(np.isfinite(image), image - center, 0.0)
    finite_deviation = deviation[np.isfinite(deviation)]
    scale = float(np.percentile(np.abs(finite_deviation), 99.5)) if finite_deviation.size else 0.0
    if not np.isfinite(scale) or scale <= 0.0:
        scale = float(np.max(np.abs(finite_deviation))) if finite_deviation.size else 0.0
    if not np.isfinite(scale) or scale <= 0.0:
        return np.full(image.shape, 128, dtype=np.uint8)
    rng = np.random.default_rng(stable_seed(label))
    normalized = 0.5 + 0.42 * deviation / scale + rng.normal(0.0, 0.01, image.shape)
    return np.clip(255.0 * normalized, 0.0, 255.0).astype(np.uint8)


def particle_from_components(name: str, components: list[dict]) -> dict:
    offsets = np.asarray([c['offset_nm'] for c in components], dtype=float)
    span = float(np.max(np.linalg.norm(offsets - offsets.mean(axis=0), axis=1))) if len(offsets) else 0.0
    largest = max(float(c['diameter_nm']) for c in components)
    hydrodynamic = max(largest, 2.0 * span + largest)
    return composite_particle(
        name=name,
        components=deepcopy(components),
        hydrodynamic_diameter_nm=hydrodynamic,
        initial_position_nm=None,
        signal_multiplier=1.0,
    )


def base_params(particle_obj: dict, z_nm: float = Z_NM) -> dict:
    params = deepcopy(PARAMS)
    params.update({
        'imaging_model': 'bright_field',
        'image_size_pixels': IMAGE_SIZE,
        'pixel_size_nm': PIXEL_SIZE_NM,
        'pupil_samples': PUPIL_SAMPLES,
        'psf_oversampling_factor': 2,
        'fps': 24.0,
        'duration_seconds': 1.0 / 24.0,
        'wavelength_nm': WAVELENGTH_NM,
        'numerical_aperture': 1.0,
        'refractive_index_medium': 1.33,
        'refractive_index_immersion': 1.518,
        'background_intensity': 1000.0,
        'read_noise_counts': 1.0,
        'camera_gain_e_per_count': 1.0,
        'shot_noise_enabled': False,
        'gaussian_noise_enabled': False,
        'fixed_pattern_gain_std': 0.0,
        'fixed_pattern_offset_counts': 0.0,
        'hot_pixel_fraction': 0.0,
        'scan_line_noise_counts': 0.0,
        'return_ideal_float_frames': True,
        'save_frame_sequence': False,
        'save_raw_frame_views': False,
        'mask_generation_enabled': False,
        'sample_environment_enabled': False,
        'sample_environment_pattern_enabled': False,
        'sample_environment_pattern': 'none',
        'background_subtraction_method': 'reference_frame',
        'z_stack_step_nm': 100.0,
        'initial_z_span_nm': 4000.0,
        'particles': [],
        'channels': None,
    })
    particle_copy = deepcopy(particle_obj)
    side_nm = IMAGE_SIZE * PIXEL_SIZE_NM
    particle_copy['motion']['initial_position_nm'] = [0.5 * side_nm, 0.5 * side_nm, float(z_nm)]
    params['particles'] = [particle_copy]
    return params


def render_particle(name: str, components: list[dict], z_nm: float = Z_NM):
    obj = particle_from_components(name, components)
    views = generate_single_frame_views(base_params(obj, z_nm))
    contrast = np.asarray(views['contrast_frame'], dtype=float)
    final = display_from_contrast(contrast, f'{name}:{z_nm}')
    return contrast, final


def shape_components() -> dict[str, list[dict]]:
    bowtie = [
        component([-120.0, -45.0, 0.0], diameter_nm=65.0, material='polystyrene'),
        component([-60.0, 0.0, 0.0], diameter_nm=65.0, material='polystyrene'),
        component([0.0, 45.0, 0.0], diameter_nm=65.0, material='polystyrene'),
        component([60.0, 0.0, 0.0], diameter_nm=65.0, material='polystyrene'),
        component([120.0, -45.0, 0.0], diameter_nm=65.0, material='polystyrene'),
    ]
    tetra = [
        component([0.0, 0.0, 90.0], diameter_nm=60.0, material='polystyrene'),
        component([85.0, 0.0, -30.0], diameter_nm=60.0, material='polystyrene'),
        component([-42.5, 73.6, -30.0], diameter_nm=60.0, material='polystyrene'),
        component([-42.5, -73.6, -30.0], diameter_nm=60.0, material='polystyrene'),
    ]
    return {
        'sphere': [component([0.0, 0.0, 0.0], diameter_nm=90.0, material='polystyrene')],
        'dimer_along_x': dimer(separation_nm=110.0, diameter_nm=75.0, material='polystyrene'),
        'linear_trimer_along_x': linear_trimer(separation_nm=95.0, diameter_nm=70.0, material='polystyrene'),
        'rod_stack_5_along_x': rod_stack(count=5, separation_nm=72.0, diameter_nm=60.0, material='polystyrene'),
        'bent_triatomic': bent_trimer(arm_separation_nm=105.0, bend_angle_deg=105.0, diameter_nm=70.0, material='polystyrene'),
        'bowtie_like': bowtie,
        'tetrahedron_3d': tetra,
    }


def rotation_matrix(axis: str, angle_rad: float) -> np.ndarray:
    c = float(np.cos(angle_rad)); s = float(np.sin(angle_rad))
    if axis == 'x': return np.array([[1,0,0],[0,c,-s],[0,s,c]], dtype=float)
    if axis == 'y': return np.array([[c,0,s],[0,1,0],[-s,0,c]], dtype=float)
    if axis == 'z': return np.array([[c,-s,0],[s,c,0],[0,0,1]], dtype=float)
    raise ValueError(axis)


def rotated_components(components: list[dict], axis: str, angle_rad: float) -> list[dict]:
    R = rotation_matrix(axis, angle_rad)
    out = []
    for comp in components:
        item = deepcopy(comp)
        item['offset_nm'] = (R @ np.asarray(comp['offset_nm'], dtype=float)).astype(float).tolist()
        out.append(item)
    return out


from PIL import Image


def save_uint8_png(path: Path, image: np.ndarray) -> None:
    arr = np.asarray(image, dtype=np.uint8)
    Image.fromarray(arr).save(path)


shapes = shape_components()
ordered = ['sphere', 'dimer_along_x', 'linear_trimer_along_x', 'rod_stack_5_along_x', 'bent_triatomic', 'bowtie_like', 'tetrahedron_3d']
image_dir = OUTPUT_DIR / 'shape_images'
array_dir = OUTPUT_DIR / 'shape_arrays'
image_dir.mkdir(parents=True, exist_ok=True)
array_dir.mkdir(parents=True, exist_ok=True)

shape_rows = []
component_manifest = {}
for name in ordered:
    contrast, image = render_particle(name, shapes[name])
    image_path = image_dir / f'{name}.png'
    contrast_path = array_dir / f'{name}_contrast.npy'
    save_uint8_png(image_path, image)
    np.save(contrast_path, contrast)
    component_manifest[name] = shapes[name]
    shape_rows.append({
        'shape': name,
        'component_count': len(shapes[name]),
        'image_png': str(image_path.relative_to(OUTPUT_DIR)),
        'contrast_npy': str(contrast_path.relative_to(OUTPUT_DIR)),
        'contrast_min': float(np.nanmin(contrast)),
        'contrast_max': float(np.nanmax(contrast)),
    })

with (OUTPUT_DIR / 'shape_image_manifest.csv').open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=list(shape_rows[0]))
    writer.writeheader()
    writer.writerows(shape_rows)
(OUTPUT_DIR / 'composite_components.json').write_text(json.dumps(component_manifest, indent=2, allow_nan=False), encoding='utf-8')

counts = [1, 2, 3, 5, 8]
scaling_rows = [{'component_count': c, 'relative_stamping_work': float(c)} for c in counts]
with (OUTPUT_DIR / 'composite_scaling.csv').open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['component_count', 'relative_stamping_work'])
    writer.writeheader()
    writer.writerows(scaling_rows)

z_step_nm = 200.0
rotation_step_rad = 0.03
orientation_shapes = ['sphere', 'dimer_along_x', 'linear_trimer_along_x', 'bent_triatomic', 'rod_stack_5_along_x', 'tetrahedron_3d']
continuous_symmetry_dim_by_shape = {
    'sphere': 3,
    'dimer_along_x': 1,
    'linear_trimer_along_x': 1,
    'rod_stack_5_along_x': 1,
    'bent_triatomic': 0,
    'tetrahedron_3d': 0,
}
singular_axes_by_shape = {
    'sphere': 'omega_x;omega_y;omega_z',
    'dimer_along_x': 'bond_axis',
    'linear_trimer_along_x': 'chain_axis',
    'rod_stack_5_along_x': 'rod_axis',
    'bent_triatomic': '',
    'tetrahedron_3d': '',
}
orientation_rows = []
for name in orientation_shapes:
    components = shapes[name]
    renders = {}
    renders['centre'], _ = render_particle(name, components)
    renders['z_minus'], _ = render_particle(name, components, z_nm=Z_NM - z_step_nm)
    renders['z_plus'], _ = render_particle(name, components, z_nm=Z_NM + z_step_nm)
    for axis in ('x', 'y', 'z'):
        renders[f'r{axis}_minus'], _ = render_particle(name, rotated_components(components, axis, -rotation_step_rad))
        renders[f'r{axis}_plus'], _ = render_particle(name, rotated_components(components, axis, rotation_step_rad))
    crlb = compute_localization_orientation_crlb(renders, NOISE_VARIANCE, PIXEL_SIZE_NM, z_step_nm, rotation_step_rad)
    symmetry_dim = continuous_symmetry_dim_by_shape[name]
    rank_prediction = predict_se3_rank_from_symmetry(symmetry_dim)
    observed_rank = int(crlb['rank'])
    orientation_rows.append({
        'shape': name,
        'sigma_xyz_nm': float(crlb['sigma_xyz_nm']),
        'sigma_omega_x_rad': float(crlb['sigma_omega_x_rad']),
        'sigma_omega_y_rad': float(crlb['sigma_omega_y_rad']),
        'sigma_omega_z_rad': float(crlb['sigma_omega_z_rad']),
        'rank': observed_rank,
        'continuous_rotational_symmetry_dim': int(symmetry_dim),
        'symmetry_nullity_lower_bound': int(rank_prediction['symmetry_nullity_lower_bound']),
        'predicted_se3_rank': int(rank_prediction['predicted_se3_rank']),
        'predicted_rotational_rank': int(rank_prediction['predicted_rotational_rank']),
        'rank_matches_symmetry_prediction': bool(observed_rank == int(rank_prediction['predicted_se3_rank'])),
        'symmetry_singular_axes': singular_axes_by_shape[name],
        'axes_singular': ';'.join(crlb.get('axes_singular', [])),
    })
with (OUTPUT_DIR / 'orientation_crlb.csv').open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=list(orientation_rows[0]))
    writer.writeheader()
    writer.writerows(orientation_rows)


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float):
        return value if np.isfinite(value) else None
    return value

(OUTPUT_DIR / 'orientation_crlb.json').write_text(json.dumps(json_safe(orientation_rows), indent=2, allow_nan=False), encoding='utf-8')
print('wrote raw E02 composite artifacts:', OUTPUT_DIR)
